# Main Figure 3

Frozen drawing logic and saved derived inputs. Outputs: `outputs/main/figure_03`. See reproduction status before release.


In [ ]:
from pathlib import Path
import os, sys
sys.dont_write_bytecode = True
_start = Path.cwd().resolve()
REPO = next((p for q in [_start, *_start.parents] for p in [q, q / "github"]
             if (p / "plot_utils" / "release_runtime.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Run from the repository root or a notebook subdirectory.")
sys.path.insert(0, str(REPO / "plot_utils"))
from release_runtime import configure
configure(REPO)


In [ ]:

from pathlib import Path
import os, sys, json, hashlib, tempfile
sys.dont_write_bytecode=True
PROJECT_ROOT=ROOT=REPO/'figure_source_data/shared'
OUTPUT_ROOT=REPO/'outputs/main/figure_03'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
RUNTIME=OUTPUT_ROOT/'runtime'; RUNTIME.mkdir(exist_ok=True)
FIG=ROOT/'fig'
for key in ['MPLCONFIGDIR','XDG_CACHE_HOME','TEMP','TMP','TMPDIR','NUMBA_CACHE_DIR']:
    os.environ[key]=str(RUNTIME)
tempfile.tempdir=str(RUNTIME)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
PRODUCTS=['GRUN','ERA5','GLDAS']
COLORS={'GRUN':'#007C78','ERA5':'#CB7935','GLDAS':'#667BA5'}
FOLDERS={'GRUN':'GRUN_try','ERA5':'ERA5_try','GLDAS':'GLADS_try'}
plt.rcParams.update({'font.family':'Arial','font.size':8,'axes.titlesize':9,
 'axes.labelsize':8,'xtick.labelsize':7,'ytick.labelsize':7,
 'axes.linewidth':.65,'lines.linewidth':1.15,'pdf.fonttype':42,
 'svg.fonttype':'none','savefig.facecolor':'white'})
SOURCE=ROOT/'Part8_change_work/figure_source_data'
source_hashes={}
def read(path):
    path=Path(path)
    source_hashes[str(path)]=hashlib.sha256(path.read_bytes()).hexdigest()
    return pd.read_csv(path)
annual=read(SOURCE/'Figure3_source_data_mean_annual.csv')
width_annual=read(SOURCE/'Figure3_source_data_width_annual.csv')
quantile=read(SOURCE/'Figure3_source_data_quantile_trends.csv')
width=read(SOURCE/'Figure3_source_data_width_trends.csv')
mean=read(ROOT/'result1.0/diagnostics/figure3_diagnostics/Figure3_source_data_global_trends.csv')
window_final=read(SOURCE/'Figure3_source_data_window.csv')
clock_parts=[]; window_parts=[]; clock_denominators=[]
for product in PRODUCTS:
    base=ROOT/FOLDERS[product]
    station=read(base/'part4_timescale_redistribution_04B/04B_global_stationarity_summary.csv')
    assert np.isclose(mean.set_index('product').loc[product,'slope'],station.iloc[0]['grid_standardized_sen'])
    tables=base/'part6_robustness_ALL_v2/tables'
    ck=read(tables/'06_03_CLOCK_CORE_ROBUSTNESS.csv')
    ck=ck[ck.metric.eq('width_q90_q10')].copy();ck['product']=product;clock_parts.append(ck)
    tr=read(tables/'06_03_ALTERNATIVE_CLOCK_AR6_TRENDS.csv')
    tr=tr[tr.metric.eq('width_q90_q10')]
    ref=tr[tr.clock.eq('tau_exact_MCoM_to_Honset_clip24')].set_index('ar6_abbrev').sen_per_decade
    for row in ck.itertuples():
        alt=tr[tr.clock.eq(row.alternative_clock)].set_index('ar6_abbrev').sen_per_decade
        pair=pd.concat([ref.rename('reference'),alt.rename('alternative')],axis=1).dropna()
        pair=pair[(pair.reference!=0)&(pair.alternative!=0)]
        assert np.isclose((np.sign(pair.reference)==np.sign(pair.alternative)).mean(),row.direction_sign_agreement)
        clock_denominators.append(dict(product=product,clock=row.alternative_clock,n_nonzero_pairs=len(pair)))
    w=read(tables/'06_04_HORIZON_GLOBAL_TRENDS.csv')
    w=w[w.metric.eq('width_q90_q10')&w.ar6_abbrev.eq('GLOBAL')].copy()
    w['product']=product;window_parts.append(w)
    expected=window_final.set_index('product').loc[product]
    for row in w.itertuples():
        assert abs(row.sen_per_decade-float(expected[str(row.window_months)]))<=.000051
clock=pd.concat(clock_parts,ignore_index=True)
window=pd.concat(window_parts,ignore_index=True)
denominators=pd.DataFrame(clock_denominators)
qplot=quantile[np.isclose(quantile['quantile'].to_numpy()[:,None],[.1,.25,.5,.75,.9]).any(axis=1)].copy()
for product in PRODUCTS:
    a=annual[annual['product'].eq(product)].sort_values('year')
    w=width_annual[width_annual['product'].eq(product)].sort_values('year')
    assert a.year.tolist()==list(range(1952,2015))==w.year.tolist()
    for q in [10,25,50,75,90]: assert np.allclose(a['Q'+str(q)],w['q'+str(q)])
    assert np.allclose(w.width_q90_q10,w.q90-w.q10)
    assert np.allclose(w.iqr,w.q75-w.q25)
assert (width.n==63).all() and (window.n_years==63).all()
assert (width.sen<0).all() and (width.sen_hi<0).all()
assert (mean.sen_low<0).all() and (mean.sen_high>0).all()
assert (window.sen_per_decade<0).all()
print('PASS: final tables, period, equal-grid means, pooled-event widths, and robustness sources verified.')
print(width.to_string(index=False))

plt.rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif']=['Arial','DejaVu Sans','Liberation Sans']
plt.rcParams['svg.fonttype']='none'


In [ ]:

import cartopy.crs as ccrs
from cartopy.io import shapereader
from shapely.geometry import shape
from matplotlib.colors import TwoSlopeNorm
grid_path=ROOT/'GRUN_try/part4_spatial_reorganization/04A_gridcell_trends_ELIGIBLE.parquet'
source_hashes[str(grid_path)]=hashlib.sha256(grid_path.read_bytes()).hexdigest()
grid=pd.read_parquet(grid_path)
regional=read(SOURCE/'Figure4_source_data_regional_width.csv')
regional=regional[regional['product'].eq('GRUN')]
geo_path=SOURCE/'Figure4_source_data_geometry.geojson'
source_hashes[str(geo_path)]=hashlib.sha256(geo_path.read_bytes()).hexdigest()
geometry=json.loads(geo_path.read_text(encoding='utf-8'))
coast_path=REPO/'assets/geography/ne_110m_coastline.shp'
coast=list(shapereader.Reader(str(coast_path)).geometries())


In [ ]:
fig=plt.figure(figsize=(7.20472440945,190/25.4),facecolor='white')
gs=fig.add_gridspec(2,3,left=.075,right=.975,bottom=.08,top=.90,wspace=.48,hspace=.60,height_ratios=[1,1.35])
def title(spec,letter,text):
    b=spec.get_position(fig)
    fig.text(b.x0-.022,b.y1+.022,letter,size=12,weight='bold',va='bottom')
    fig.text(b.x0,b.y1+.022,text,size=9,weight='bold',va='bottom')
def clean(ax):
    ax.spines[['top','right']].set_visible(False)
    ax.tick_params(width=.6,length=3)
def zero(ax,axis='y'):
    (ax.axhline if axis=='y' else ax.axvline)(0,color='.55',lw=.7,ls='--',zorder=0)
handles=[Line2D([],[],color=COLORS[p],marker='o',lw=1.3,ms=4,label=p) for p in PRODUCTS]
fig.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,.99),ncol=3,frameon=False)
fig.text(.985,.97,'1952–2014',ha='right',fontsize=8,color='.3')
unit='months decade⁻¹'



In [ ]:
title(gs[0,0],'a','Annual propagation-time distributions')
ag=gs[0,0].subgridspec(3,1,hspace=.18)
for i,p in enumerate(PRODUCTS):
    ax=fig.add_subplot(ag[i]);clean(ax)
    d=annual[annual['product'].eq(p)].sort_values('year');x=d.year.to_numpy();color=COLORS[p]
    ax.fill_between(x,d.Q10,d.Q90,color=color,alpha=.15,lw=0)
    ax.fill_between(x,d.Q25,d.Q75,color=color,alpha=.27,lw=0)
    ax.plot(x,d.Q50,color=color,lw=1.1)
    ax.plot(x,d.equal_grid_PT,color='#202020',lw=.95,ls='--')
    ax.text(.02,.89,p,color=color,weight='bold',transform=ax.transAxes,fontsize=8)
    ax.set_xlim(1952,2014);ax.set_ylim(3.5,19.5);ax.set_yticks([5,10,15])
    ax.set_xticks([1960,1980,2000,2014])
    if i<2:ax.tick_params(labelbottom=False)
    else:ax.set_xlabel('Year')
    if i==1:ax.set_ylabel('Propagation time (months)')
b=gs[0,0].get_position(fig)
fig.legend(handles=[Patch(facecolor='.85',label='Q10–Q90'),Patch(facecolor='.65',label='IQR'),
 Line2D([],[],color='.2',label='Q50'),Line2D([],[],color='.2',ls='--',label='Equal-grid mean')],
 loc='upper left',bbox_to_anchor=(b.x0-.005,b.y0-.042),ncol=2,frameon=False,fontsize=6.5,
 columnspacing=.8,handlelength=1.8)



In [ ]:

title(gs[0,1],'b','Quantile trend distributions')
bg=gs[0,1].subgridspec(3,1,hspace=.23)
for i,p in enumerate(PRODUCTS):
    ax=fig.add_subplot(bg[i]);clean(ax);zero(ax,'x')
    d=qplot[qplot['product'].eq(p)].sort_values('quantile')
    base=np.array(matplotlib.colors.to_rgb(COLORS[p]))
    for j,row in enumerate(d.itertuples()):
        color=tuple(base*(1-j*.14)+np.ones(3)*j*.14)
        ax.barh(j,row.ci_high-row.ci_low,left=row.ci_low,height=.64,color=color,alpha=.65,edgecolor='none')
        ax.plot([row.slope,row.slope],[j-.27,j+.27],color=color,lw=2.2)
    ax.set_yticks(range(5),['Q10','Q25','Q50','Q75','Q90'],fontsize=6)
    ax.set_ylim(4.7,-.8);ax.set_xlim(-.205,.185)
    ax.set_xticks([-.2,-.1,0,.1])
    ax.text(.02,.92,p,ha='left',va='top',transform=ax.transAxes,color=COLORS[p],weight='bold',fontsize=8)
    if i<2:ax.tick_params(labelbottom=False)
    else:ax.set_xlabel('Sen slope ('+unit+')')
pos=gs[0,1].get_position(fig)
fig.text(pos.x0,pos.y0-.048,'Bands: existing pointwise 95% bootstrap intervals',fontsize=6.5)


In [ ]:
title(gs[0,2],'c','Distribution-width trends')
ax=fig.add_subplot(gs[0,2]);clean(ax);zero(ax,'x')
metrics=['width_q90_q10','iqr','mad']
for i,p in enumerate(PRODUCTS):
    d=width[width['product'].eq(p)].set_index('metric').loc[metrics]
    ax.barh(np.arange(3)+(i-1)*.2,d.sen,height=.14,color=COLORS[p],alpha=.2)
    ax.errorbar(d.sen,np.arange(3)+(i-1)*.2,
        xerr=np.vstack([d.sen-d.sen_lo,d.sen_hi-d.sen]),fmt='o',color=COLORS[p],
        ms=4,elinewidth=1,capsize=2)
ax.set_yticks(range(3),['Q90 − Q10','IQR','MAD']);ax.set_ylim(2.5,-.5)
ax.set_xlabel('Sen slope ('+unit+')');ax.set_xlim(-.37,.035)
ax.text(.03,.04,'95% Theil–Sen intervals',transform=ax.transAxes,fontsize=6.5)



In [ ]:
title(gs[1,0],'d','Spatial reorganization (GRUN)')
dg=gs[1,0].subgridspec(2,1,height_ratios=[1,1],hspace=.65)
def mapbase(spec,subtitle):
    ax=fig.add_subplot(spec,projection=ccrs.Robinson())
    ax.set_global()
    ax.set_extent([-180,180,-60,85],crs=ccrs.PlateCarree())
    ax.add_geometries(coast,ccrs.PlateCarree(),facecolor='none',edgecolor='.35',linewidth=.25,zorder=4)
    ax.spines['geo'].set_linewidth(.4)
    ax.set_title(subtitle,loc='left',fontsize=8,pad=3)
    return ax
ax=mapbase(dg[0],'d1  GRUN grid-year mean PT trend')
z=grid.pivot(index='lat',columns='lon',values='sen_months_per_decade').sort_index().sort_index(axis=1)
norm=TwoSlopeNorm(vmin=-1,vcenter=0,vmax=1)
im=ax.pcolormesh(z.columns,z.index,z.values,transform=ccrs.PlateCarree(),
                cmap='RdBu_r',norm=norm,shading='nearest',rasterized=True)
cb=fig.colorbar(im,ax=ax,orientation='horizontal',fraction=.04,pad=.035,aspect=45,extend='both')
cb.set_label('Mean PT trend ('+unit+')',fontsize=7,labelpad=1)
cb.ax.tick_params(labelsize=6,pad=1,length=2)
ax=mapbase(dg[1],'d2  GRUN AR6 width trend')
norm2=TwoSlopeNorm(vmin=-.6,vcenter=0,vmax=.6)
vals=regional.set_index('ar6').sen_dec
for feature in geometry['features']:
    if feature['properties'].get('Type')!='Land': continue
    key=feature['properties']['Acronym']
    color=plt.cm.RdBu_r(norm2(vals[key])) if key in vals.index else '#e6e6e6'
    ax.add_geometries([shape(feature['geometry'])],ccrs.PlateCarree(),
                     facecolor=color,edgecolor='.4',linewidth=.3)
sm=plt.cm.ScalarMappable(norm=norm2,cmap='RdBu_r')
cb=fig.colorbar(sm,ax=ax,orientation='horizontal',fraction=.04,pad=.035,aspect=45,extend='both')
cb.set_label('Q90 − Q10 trend ('+unit+')',fontsize=7,labelpad=1)
cb.ax.tick_params(labelsize=6,pad=1,length=2)


In [ ]:

title(gs[1,1],'e','Annual distribution-width evolution')
ax=fig.add_subplot(gs[1,1]);clean(ax)
smooth=[]
for p in PRODUCTS:
    d=width_annual[width_annual['product'].eq(p)].sort_values('year').copy()
    d['running_mean_5yr']=d.width_q90_q10.rolling(5,center=True,min_periods=5).mean()
    smooth.append(d[['product','year','width_q90_q10','running_mean_5yr']])
    ax.plot(d.year,d.width_q90_q10,color=COLORS[p],lw=.55,alpha=.23)
    ax.plot(d.year,d.running_mean_5yr,color=COLORS[p],lw=1.8)
smooth=pd.concat(smooth,ignore_index=True)
ax.set_xlim(1952,2014);ax.set_xticks([1960,1980,2000,2014])
ax.set_xlabel('Year');ax.set_ylabel('Annual Q90 − Q10 (months)')
ax.text(.03,.035,'Thin: annual values\nThick: 5-year mean (visual guide)',transform=ax.transAxes,fontsize=7)


In [ ]:
title(gs[1,2],'f','Reference and window sensitivity')
fg=gs[1,2].subgridspec(2,1,hspace=.53,height_ratios=[1.1,1])
clocks=['tau_earliest_onset','tau_nearest_onset','tau_strongest_onset','tau_strongest_peak',
        'tau_exact_MCoM_to_Honset','tau_exact_CoM_to_CoM','tau_exact_CoM_to_CoM_clip24']
labels=['Earliest onset','Nearest onset','Strongest onset','Strongest peak',
        'MCoM → H onset (full)','MCoM → HCoM (full)','MCoM → HCoM (24 mo)']
ax=fig.add_subplot(fg[0])
matrix=clock.pivot(index='alternative_clock',columns='product',values='direction_sign_agreement').loc[clocks,PRODUCTS]*100
im=ax.imshow(matrix,aspect='auto',vmin=0,vmax=100,cmap='Blues')
ax.set_xticks(range(3),PRODUCTS);ax.set_yticks(range(7),labels,fontsize=6.7)
ax.tick_params(length=0)
ax.set_title('f1  Regional sign consistency across references',loc='left',fontsize=8,pad=6)
for i in range(7):
 for j in range(3):ax.text(j,i,f'{matrix.iloc[i,j]:.0f}%',ha='center',va='center',fontsize=8,color='white' if matrix.iloc[i,j]>65 else '#202020')
cb=fig.colorbar(im,ax=ax,orientation='horizontal',fraction=.04,pad=.12,aspect=45)
cb.set_label('AR6 sign agreement vs 24-month CoM (%)',fontsize=7,labelpad=1)
cb.ax.tick_params(labelsize=6,length=2,pad=1)
ax=fig.add_subplot(fg[1])
matrix=window.pivot(index='window_months',columns='product',values='sen_per_decade').loc[[18,21,24,27,30,36],PRODUCTS]
im=ax.imshow(matrix,aspect='auto',cmap='RdBu_r',norm=TwoSlopeNorm(vmin=-.18,vcenter=0,vmax=.18))
ax.set_xticks(range(3),PRODUCTS);ax.set_yticks(range(6),['18 mo','21 mo','24 mo','27 mo','30 mo','36 mo'])
ax.tick_params(length=0)
ax.set_title('f2  Global width trend across windows',loc='left',fontsize=8,pad=6)
for i in range(6):
 for j in range(3):ax.text(j,i,f'{matrix.iloc[i,j]:+.3f}',ha='center',va='center',fontsize=8,color='white' if matrix.iloc[i,j]<-.115 else '#202020')
cb=fig.colorbar(im,ax=ax,orientation='horizontal',fraction=.04,pad=.13,aspect=45)
cb.set_label('Width trend ('+unit+'); point estimates',fontsize=7,labelpad=1)
cb.ax.tick_params(labelsize=6,length=2,pad=1)
fig.text(.065,.022,'Main widths pool events; d2 and f use distributions of grid-year means. Blue map values indicate shortening / narrowing; grey AR6 areas lack estimates.',fontsize=7,color='.3')


In [ ]:

FIGURE_NUMBER=3
FROZEN_TITLES={'a': 'CoM mean trends are weak across products', 'b': 'Opposing quantile trends compress distributions', 'c': 'Widths narrow across products', 'd': 'Spatial trends reveal heterogeneous reorganization', 'e': 'Distribution widths contract through time', 'f': 'Reference agreement varies across regions'}
from matplotlib.text import Text
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import textwrap
OUTDIR=Path(OUT if FIGURE_NUMBER==6 else OUTPUT_ROOT)
QA=OUTDIR/'qa';QA.mkdir(exist_ok=True)
def numerical_artists():
    result={}
    def add(key,value):
        a=np.ma.asarray(value)
        if a.dtype.kind in 'biufc':
            v=np.ascontiguousarray(a.filled(np.nan) if np.ma.isMaskedArray(a) and a.dtype.kind in 'fc' else a)
            result[key]={'shape':list(v.shape),'sha256':hashlib.sha256(v.tobytes()).hexdigest()}
    for j,a in enumerate(fig.axes):
        for k,l in enumerate(a.lines):add(f'{j}/line{k}/x',l.get_xdata());add(f'{j}/line{k}/y',l.get_ydata())
        for k,l in enumerate(a.collections):
            if l.get_array() is not None:add(f'{j}/collection{k}/array',l.get_array())
            if hasattr(l,'get_offsets'):add(f'{j}/collection{k}/offsets',l.get_offsets())
            if hasattr(l,'get_segments'):
                for h,v in enumerate(l.get_segments()):add(f'{j}/collection{k}/segment{h}',v)
            if l.__class__.__name__ in ('PolyCollection','FillBetweenPolyCollection'):
                for h,v in enumerate(l.get_paths()):add(f'{j}/collection{k}/path{h}',v.vertices)
        for k,l in enumerate(a.images):add(f'{j}/image{k}',l.get_array())
        for k,l in enumerate(a.patches):
            if isinstance(l,matplotlib.patches.Rectangle):add(f'{j}/bar{k}',[l.get_x(),l.get_y(),l.get_width(),l.get_height()])
    return result
fig.canvas.draw()
before=numerical_artists()
contracts=json.loads((REPO/'assets/other/panel_contracts.json').read_text(encoding='utf-8'))
oldtitles={r[2].replace('clock','reference').replace('Clock','Reference') for r in contracts if r[0]==FIGURE_NUMBER}
notes=[]
for t in list(fig.findobj(Text)):
    s=t.get_text()
    if s in oldtitles or s in list('abcdef'):
        t.set_text('');continue
    if len(s)>180:
        notes.append(s);t.set_text('');continue
    s=s.replace('decade$^{-1}$','decade⁻¹').replace(r'$\beta$','β').replace('$R^2$','R²')
    t.set_text(s);t.set_fontfamily('DejaVu Sans');t.set_fontsize(5.5 if FIGURE_NUMBER in (1,2) else 6)
    if t.get_rotation():t.set_rotation_mode('anchor')
plt.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial','Liberation Sans'],
    'font.size':6,'axes.labelsize':6,'xtick.labelsize':5.5,'ytick.labelsize':5.5,
    'legend.fontsize':5.5,'pdf.fonttype':42,'svg.fonttype':'none'})
DIV=plt.get_cmap('RdBu_r')
for ax in fig.axes:
    cb=getattr(ax,'_colorbar',None)
    for sp in ax.spines.values():sp.set_linewidth(.45);sp.set_edgecolor('#657076')
    ax.tick_params(width=.45,length=2,pad=2,labelsize=5.5)
    ax.xaxis.label.set_fontsize(6);ax.yaxis.label.set_fontsize(6)
    ax.title.set_fontsize(6)
    if cb is not None:
        cb.outline.set_linewidth(.35)
        ax.tick_params(length=1.5,pad=1)
        ax.set_axes_locator(None)
        p=ax.get_position()
        if cb.orientation=='horizontal':ax.set_position([p.x0,p.y0,p.width,min(p.height,2.5/(72*fig.get_figheight()))])
        else:ax.set_position([p.x0,p.y0,min(p.width,2.5/(72*fig.get_figwidth())),p.height])
    for artist in list(ax.images)+list(ax.collections):
        norm=getattr(artist,'norm',None)
        if norm is not None and norm.vmin is not None and norm.vmax is not None and norm.vmin<0<norm.vmax:
            assert np.isclose(abs(norm.vmin),abs(norm.vmax)), 'Original diverging limits require review'
            artist.set_cmap(DIV);artist.set_norm(TwoSlopeNorm(vmin=norm.vmin,vcenter=0,vmax=norm.vmax))
    for line in ax.lines:
        if line.get_linestyle() in ('--',':'):
            x=np.asarray(line.get_xdata());y=np.asarray(line.get_ydata())
            if (len(x)>0 and np.all(x==0)) or (len(y)>0 and np.all(y==0)):
                line.set_color('#8B9298');line.set_linewidth(.5);line.set_linestyle('--')
        if line.get_markersize()>4:line.set_markersize(3.5)
        if line.get_linewidth()>1.8:line.set_linewidth(1.5)
    if hasattr(ax,'projection'):
        for col in ax.collections:
            if col.__class__.__name__=='FeatureArtist':col.set_linewidth(.25)
        ax.spines['geo'].set_linewidth(.45)

if FIGURE_NUMBER==1:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='24 mo':t.set_x(.62);t.set_transform(ax.transAxes);t.set_y(.07);t.set_ha('right')
            if t.get_text().startswith('d1  '):t.set_position((0,1.02));t.set_va('bottom')
            if t.get_text().startswith('Common 36-month'):
                notes.append(t.get_text());t.set_text('Common 36-month\ncoverage cohort');t.set_position((.04,.06));t.set_fontsize(5)
if FIGURE_NUMBER==2:
    for ax in fig.axes:
        if ax.get_xlabel()=='Months relative to meteorological anchor':ax.set_xlabel('Months from meteorological anchor',fontsize=6)
        for t in ax.texts:
            if t.get_text().startswith('Negative ='):notes.append(t.get_text());t.set_text('')
            if '95% block CI' in t.get_text():t.set_position((.03,1.02));t.set_va('bottom');t.set_fontsize(5)
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb is not None and 'hexagon' in ax.get_xlabel():
            from matplotlib.ticker import FixedLocator,FixedFormatter,NullLocator
            ax.xaxis.set_major_locator(FixedLocator([1,10,100,1000]));ax.xaxis.set_major_formatter(FixedFormatter(['1','10','100','1000']));ax.xaxis.set_minor_locator(NullLocator())
if FIGURE_NUMBER==3:
    for leg in list(fig.legends):
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            handles=leg.legend_handles;labels=[t.get_text() for t in leg.get_texts()]
            pos=gs[0,0].get_position(fig);leg.remove()
            fig.legend(handles,labels,ncol=4,frameon=False,fontsize=5.2,loc='upper left',bbox_to_anchor=(pos.x0-.005,pos.y0-.038),handlelength=1.2,columnspacing=.8)
    maps=[a for a in fig.axes if hasattr(a,'projection')]
    for a in maps:
        a.title.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Main widths pool events'):
            t.set_text('Main widths: pooled events. d2/f: grid-year means. Grey: no estimate.');t.set_fontsize(5.5)
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Thin: annual'):
                t.set_position((.02,1.01));t.set_va('bottom');t.set_text('Thin: annual; thick: 5-year visual mean');t.set_fontsize(5)
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='horizontal':
            p=ax.get_position();ax.set_position([p.x0,p.y0-.01,p.width,p.height]);cb.set_ticks([cb.norm.vmin,0,cb.norm.vmax] if cb.norm.vmin<0 else [0,50,100])
        if ax.images:
            p=ax.get_position();ax.set_position([p.x0+.04,p.y0,p.width-.04,p.height]);ax.tick_params(axis='y',labelsize=5)
if FIGURE_NUMBER==4:
    for leg in fig.legends:leg.set_bbox_to_anchor((.5,.045))
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text().replace('Parent number','Antecedent count').replace('Many-to-one','Multiple antecedents')
            if '  ρ: ' in s:
                s=s.replace('  ρ: ','\nρ: ');t.set_position((.01,1.03));t.set_va('bottom');t.set_fontsize(5.2)
            if s.startswith('a1  '):
                t.set_text('');p=ax.get_position();fig.text(p.x0,p.y1+.013,'a1  Event scale · GRUN',fontsize=5.5,fontweight='bold')
            else:t.set_text(s)
            if not ax.axison:t.set_fontsize(5.2)
        if ax.get_ylabel()=='TP slope':ax.set_ylabel('PT slope',fontsize=6)
        cb=getattr(ax,'_colorbar',None)
        if cb is not None:cb.set_ticks([-1,0,1])
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        for t in ax.texts:
            s=t.get_text()
            if s.startswith('Envelope spans') or s.startswith('Filled:'):
                t.set_position((0,-.22));t.set_ha('left');t.set_va('top');t.set_fontsize(5.2)
            if s=='Severity decile':t.set_x(-.29);t.set_fontsize(5.5)
            if 'median β < 0' in s:
                t.set_position((.0,-.32));t.set_ha('left');t.set_va('top');t.set_text(s.replace('\n','; ',1));t.set_fontsize(5.2);t.set_bbox(None)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._loc=2
            for cax in fig.axes:
                cb=getattr(cax,'_colorbar',None)
                if cb and cb.orientation=='horizontal':
                    p=ax.get_position();cax.set_position([p.x0,p.y0-.105,p.width,2.5/(72*fig.get_figheight())])
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text()=='threshold floor':t.set_text('AUC < 0.60: L* = 0');t.set_x(.34);t.set_fontsize(5.5)
            if t.get_text().startswith('Δ ='):
                t.set_x(1.0);t.set_fontsize(5);t.set_bbox(None)
                if ax.get_title()=='Retrospective PT':
                    t.set_x(.02);t.set_ha('left');t.set_text(t.get_text().replace('Δ = ','Δ '))
            if t.get_text() in ('GRUN','ERA5','GLDAS') and ax in fig.axes[12:]:
                t.set_position((.50,.84));t.set_fontsize(5.5)
    for t in fig.texts:
        if t.get_text().startswith('Points: temporal'):t.set_y(.025);t.set_fontsize(5)


for ax in fig.axes:
    leg=ax.get_legend()
    if leg:
        for t in leg.get_texts():t.set_fontsize(5)
    for t in ax.texts:
        if len(t.get_text())>36:t.set_text(textwrap.fill(t.get_text(),36,break_long_words=False));t.set_fontsize(5)
    if len(ax.get_xlabel())>32:ax.set_xlabel(textwrap.fill(ax.get_xlabel(),32,break_long_words=False),fontsize=5.5)
if FIGURE_NUMBER==3:
    for t in fig.texts:
        if t.get_text().startswith(('f1 ','f2 ','Bands:')):t.set_text(textwrap.fill(t.get_text(),34,break_long_words=False));t.set_fontsize(5)
    for leg in fig.legends:
        if any(t.get_text()=='Q10–Q90' for t in leg.get_texts()):
            h=leg.legend_handles;l=[t.get_text() for t in leg.get_texts()];p=gs[0,0].get_position(fig);leg.remove();fig.legend(h,l,ncol=2,frameon=False,fontsize=5,loc='upper left',bbox_to_anchor=(p.x0,p.y0-.04))
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.94,p.y0,p.width,p.height]);ax.yaxis.label.set_fontsize(5)
        for t in ax.texts:
            if t.get_text()=='Severity decile':t.set_x(-.40)
        if hasattr(ax,'projection'):
            leg=ax.get_legend()
            if leg:leg.set_bbox_to_anchor((0,-.10));leg._ncols=1
            for t in ax.texts:
                if 'median β' in t.get_text():t.set_y(-.45)
if FIGURE_NUMBER==6:
    for ax in fig.axes:
        if ax.get_xlabel().startswith('Sen slope'):ax.set_xlabel('Sen slope\n(months decade⁻¹)',fontsize=5)
        cb=getattr(ax,'_colorbar',None)
        if cb:
            p=ax.get_position();ax.set_position([p.x0,p.y0-.016,p.width,p.height])
        if ax.get_title()=='Retrospective PT':ax.set_xticks([-.08,0,.04])
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.078))
    for t in fig.texts:
        if t.get_text().startswith('Δ = high'):t.set_y(.046);t.set_text('Δ = high − low; bars:\ncomponent 95% CIs');t.set_fontsize(5)
        if t.get_text().startswith('Points: temporal'):t.set_text('Points: temporal Sen slopes;\nbars: 95% Sen intervals');t.set_y(.035);t.set_fontsize(5)


if FIGURE_NUMBER==3:
    for ax in fig.axes:
        if ax.get_title().startswith(('f1 ','f2 ')):ax.set_title(textwrap.fill(ax.get_title(),29),loc='left',fontsize=5,pad=8)
    for t in fig.findobj(Text):
        if t.get_text().startswith(('f1 ','f2 ')):t.set_text(textwrap.fill(t.get_text(),29));t.set_fontsize(5)
        if t.get_text().startswith('Bands:'):t.set_y(t.get_position()[1]-.027)
if FIGURE_NUMBER==5:
    for ax in fig.axes:
        cb=getattr(ax,'_colorbar',None)
        if cb and cb.orientation=='vertical':
            p=ax.get_position();ax.set_position([.923,p.y0,p.width,p.height]);ax.yaxis.labelpad=1
        leg=ax.get_legend()
        if leg and any(t.get_text() in ('Q50','Min. events') for t in leg.get_texts()):leg.set_bbox_to_anchor((0,-.18));leg._loc=2
    for ax in fig.axes:
        for t in ax.texts:
            if t.get_text().startswith('Filled:'):t.set_y(-.34)
if FIGURE_NUMBER==6:
    for leg in fig.legends:leg.set_bbox_to_anchor((bottom[0].get_position(fig).x0,.095))

if FIGURE_NUMBER in (1,2,3):specs=[gs[0,0],gs[0,1],gs[0,2],gs[1,0],gs[1,1],gs[1,2]]
elif FIGURE_NUMBER==4:specs=[gs[0,0],gs[0,1],gs[1,0],gs[1,1]]
elif FIGURE_NUMBER==5:specs=[outer[0,0],outer[0,1],outer[0,2],outer[1,0],outer[1,1],outer[1,2]]
else:specs=top+bottom
headers=[]
for letter,spec in zip('abcdef',specs):
    b=spec.get_position(fig);h=fig.get_figheight()*72;w=fig.get_figwidth()*72
    width_chars=33 if FIGURE_NUMBER in (1,2) else 28
    title_offset=9 if FIGURE_NUMBER in (1,2) else 24 if FIGURE_NUMBER==4 else 18
    txt=textwrap.fill(FROZEN_TITLES[letter],width_chars,break_long_words=False)
    t=fig.text(b.x0,b.y1+title_offset/h,txt,ha='left',va='bottom',fontsize=6.5,fontweight='normal',linespacing=1.4,color='#20272B')
    fig.text(b.x0-10/w,b.y1+title_offset/h,letter,ha='left',va='bottom',fontsize=8,fontweight='bold',color='#20272B')
    headers.append(t)
fig.canvas.draw()
after=numerical_artists()
assert before==after,'A scientific artist changed during styling'
(QA/'numerical_artist_invariance.json').write_text(json.dumps({'status':'PASS','entries':len(before),'before':before,'after':after},indent=2),encoding='utf-8')
(QA/'retained_caption_notes.json').write_text(json.dumps(notes,ensure_ascii=False,indent=2),encoding='utf-8')
from audit_panel_alignment import require_matplotlib_panel_alignment
eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None and not hasattr(a,'projection')]
if len(eligible)<2:eligible=[a for a in fig.axes if a.axison and getattr(a,'_colorbar',None) is None]
ids=[f'plot{i+1}' for i in range(len(eligible))]
rows={};cols={}
for ident,ax in zip(ids,eligible):
    p=ax.get_position();rows.setdefault((round(p.y0,5),round(p.y1,5)),[]).append(ident);cols.setdefault((round(p.x0,5),round(p.x1,5)),[]).append(ident)
rg=[v for v in rows.values() if len(v)>1];cg=[v for v in cols.values() if len(v)>1]
ex=[]
for group in cg:
    if len(group)>=3:
        parent_ids={str(eligible[ids.index(k)].get_subplotspec().get_gridspec()) for k in group}
        ex.append({'panels':group,'checks':['vertical-gutter'],'reason':'Column spans distinct main-panel rows and nested stacks; outer and inner gutters intentionally differ. Stack spacing remains uniform within each original nested GridSpec.'})
for group in rg:
    if len(group)>=3:
        widths=[eligible[ids.index(k)].get_position().width for k in group]
        if np.ptp(widths)>1e-4:
            ex.append({'panels':group,'checks':['panel-width','horizontal-gutter'],'reason':'Nested composites intentionally allocate widths to distinct quantities; within-product repeats are compared separately.'})
require_matplotlib_panel_alignment(fig,axes=eligible,panel_ids=ids,row_groups=rg,column_groups=cg,
    exemptions=ex,require_panel_labels=False,json_out=QA/'alignment.json',overlay_svg=QA/'alignment.svg',strict=True)
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.png',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.pdf',dpi=450,facecolor='white')
fig.savefig(OUTDIR/f'Figure{FIGURE_NUMBER}_v2.0.svg',dpi=450,facecolor='white')
fig.canvas.draw()
meta=[{'axis':j,'bounds':list(a.get_position().bounds),'title':a.get_title(),'xlabel':a.get_xlabel(),'ylabel':a.get_ylabel(),
       'texts':[t.get_text() for t in a.texts]} for j,a in enumerate(fig.axes)]
(QA/'axes.json').write_text(json.dumps(meta,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Figure {FIGURE_NUMBER}: exported, {len(before)} numerical artist arrays unchanged.')
plt.close(fig)
